# 02: DataFrame Joins

**Exam objective (from Data Transformation and Modeling domain):** 
Combine DataFrames with operations such as Inner join, left join, broadcast join, multiple keys, cross join, union, and union all.

**Scope:** All standard join types in PySpark, plus Spark-specific 
optimizations (broadcast join) and edge cases (null keys, duplicate rows from joins). Union operations included.

In [0]:
from pyspark.sql import Row

# Customers table — 5 customers
customers_data = [
    Row(customer_id="C001", name="Alice",   region="north"),
    Row(customer_id="C002", name="Bob",     region="south"),
    Row(customer_id="C003", name="Carol",   region="east"),
    Row(customer_id="C004", name="David",   region="west"),
    Row(customer_id="C005", name="Eve",     region="north"),
]
customers = spark.createDataFrame(customers_data)

# Orders table — 6 orders, but with some intentional structure
orders_data = [
    Row(order_id="1001", customer_id="C001", amount=149.99, order_date="2026-07-15"),
    Row(order_id="1002", customer_id="C001", amount=89.50,  order_date="2026-07-16"),  # C001 has 2 orders
    Row(order_id="1003", customer_id="C002", amount=245.00, order_date="2026-07-16"),
    Row(order_id="1004", customer_id="C003", amount=52.25,  order_date="2026-07-17"),
    Row(order_id="1005", customer_id="C007", amount=99.00,  order_date="2026-07-18"),  # C007 doesn't exist in customers
    Row(order_id="1006", customer_id=None,   amount=175.00, order_date="2026-07-18"),  # null customer_id
]
orders = spark.createDataFrame(orders_data)

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")

print("Customers:")
display(customers)

print("Orders:")
display(orders)

## Sample data structure

Two tables representing a customer-orders relationship, with deliberate edge cases:

- Customers: C001-C005, five customers across four regions.
- Orders: Six orders, with the following non-trivial cases:
  - Customer C001 has 2 orders (join will duplicate customer row).
  - Customers C004 and C005 have 0 orders (dropped from inner join).
  - Order 1005 references non-existent customer C007 (dropped from inner join).
  - Order 1006 has null customer_id (special handling on join).

Each edge case demonstrates a specific join behavior in the steps that follow.

In [0]:
# inner join

# form 1 -> deduplicates the join column
inner1 = customers.join(orders, on="customer_id", how="inner")

# form 2, when matching columns do not share the same name -> does not depulicate the join column (end up with both join columns)
inner2 = customers.join(orders, on=customers.customer_id == orders.customer_id, how="inner")

display(inner1.orderBy("customer_id", "order_id"))

In [0]:
# compare forms
print("Form 1 columns: ", inner1.columns)
print("Form 2 columns: ", inner2.columns)

## Inner join

Returns rows where the join key exists in both tables. Standard SQL join semantics.

**PySpark syntax:**
- `df1.join(df2, on="col", how="inner")` — clean form when column names match; result has one join column.
- `df1.join(df2, on=df1.col == df2.col, how="inner")` — explicit form needed when column names differ; result has both join columns (potentially duplicated).

**Null-key behavior:** Nulls in join keys are not matched. A row with `customer_id = null` will be excluded from the inner join even if another table also has a row with `customer_id = null`. This is standard SQL semantics (`null = null` evaluates to `null`, not `true`).

**Row duplication:** When one side has multiple matches for a join key, the other side's row is repeated for each match. In our data, C001 has 2 orders, so Alice's row appears twice in the join output — once per matching order.

In [0]:
# left join
left = customers.join(orders, on="customer_id", how="left") # same as "left_outer"
display(left.orderBy("customer_id", "order_id"))

In [0]:
# right join
right = customers.join(orders, on="customer_id", how="right") # same as "right_outer"
display(right.orderBy("customer_id", "order_id"))

In [0]:
# full join
full = customers.join(orders, on="customer_id", how="full") # same as "outer" or "full_outer"
display(full.orderBy("customer_id", "order_id")) 

## Left, right, full outer joins

**Left join** (`how="left"` or `how="left_outer"`): keeps all rows from the left side. Right-side columns are null when there's no match.
- Use case: "For each customer, get their orders (if any)." You want every customer represented, whether or not they've ordered.

**Right join** (`how="right"` or `how="right_outer"`): keeps all rows from the right side.
- Equivalent to a left join with the sides swapped.
- Rarely used in practice; convention favors left joins.

**Full outer join** (`how="outer"` or `how="full"` or `how="full_outer"`): keeps all rows from both sides.
- Use case: reconciliation — finding rows that exist in one dataset but not the other (or both). Common in data quality checks.

**All three preserve orphaned rows differently than inner join:**
- Inner: drop orphans from both sides.
- Left: keep orphans from left, drop orphans from right.
- Right: keep orphans from right, drop orphans from left.
- Outer: keep orphans from both.

In [0]:
import pyspark.sql.functions as F

# Orders whose customer_id doesn't exist in customers
orphaned_orders = (customers.join(orders, on="customer_id", how="outer")
    .filter(F.col("name").isNull())
)
display(orphaned_orders)

In [0]:
# joining on multiple keys

inventory_data = [
    Row(warehouse_id="W1", sku="S001", quantity=100),
    Row(warehouse_id="W1", sku="S002", quantity=50),
    Row(warehouse_id="W2", sku="S001", quantity=75),   # same SKU, different warehouse
    Row(warehouse_id="W2", sku="S003", quantity=200),
]
inventory = spark.createDataFrame(inventory_data)

restock_orders_data = [
    Row(warehouse_id="W1", sku="S001", restock_qty=50),
    Row(warehouse_id="W1", sku="S002", restock_qty=25),
    Row(warehouse_id="W2", sku="S001", restock_qty=100),
    Row(warehouse_id="W2", sku="S004", restock_qty=30),  # SKU not in inventory
]
restock_orders = spark.createDataFrame(restock_orders_data)

display(inventory)
display(restock_orders)

In [0]:
# join on more than one key
joined = inventory.join(
    restock_orders, 
    on=["warehouse_id", "sku"],
    how="inner"
)

display(joined.orderBy("warehouse_id", "sku"))

## Joins on multiple keys

**Syntax:**
- `df1.join(df2, on=["k1", "k2"], how="...")` — list of column names when they match on both sides.
- `df1.join(df2, on=(df1.k1 == df2.k1) & (df1.k2 == df2.k2), how="...")` 
  — explicit conditions when names differ or non-equality needed.

**Why composite keys matter:**

Real data often has natural composite keys — a value is unique only within a scope (SKU within warehouse, order line within order, enrollment within school). Joining on only one column of the composite key produces a fan-out: rows explode when values that repeat across scopes match each other cross-scope.

**The failure mode:** if the join key isn't unique on at least one side, you get more output rows than intended. Aggregations become wrong because rows are double-counted. The fix is to include all components of the natural key in the join.

**How to verify:** check row counts before and after the join. If the output row count exceeds either input's row count *and* you didn't expect a many-to-many relationship, you have a fan-out. Investigate the join key.

In [0]:
# Check whether a column is a valid uniqueness key
inventory.groupBy("sku").count().filter(F.col("count") > 1).show()

In [0]:
inventory.groupBy("warehouse_id", "sku").count().filter(F.col("count") > 1).show()

In [0]:
from pyspark.sql.functions import broadcast

# force a broadset join by hinting the small side
enriched = orders.join(broadcast(customers), on="customer_id", how="inner")
display(enriched.orderBy("customer_id", "order_id"))

In [0]:
# Check current threshold -> does not work on serverless
# print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

In [0]:
# Without hint — Spark decides based on statistics
plain_join = orders.join(customers, on="customer_id", how="inner")
plain_join.explain()

print("=" * 80)

# With explicit hint
broadcast_join = orders.join(broadcast(customers), on="customer_id", how="inner")
broadcast_join.explain()

## Broadcast join

Optimization that eliminates the shuffle for one side of a join by sending (broadcasting) a small table to every executor. Each executor joins its local partition of the large side against the full broadcast copy of the small side.

**When it fits:**
- One side small (typically under 100 MB, often under 10 MB).
- The other side large.
- Join is any type except cross join.

**Two ways to enable:**
1. **Automatic**: Spark broadcasts if one side is smaller than `spark.sql.autoBroadcastJoinThreshold` (default 10 MB) and statistics are available.
2. **Explicit hint**: `broadcast(df)` forces a broadcast join regardless of size.
```python
   from pyspark.sql.functions import broadcast
   large.join(broadcast(small), on="key", how="inner")
```

**When it hurts:**
- Broadcasting a large table causes OOM on executors and driver.
- The threshold exists for a reason — broadcasting anything large is usually wrong.

**How to verify:**
`df.explain()` shows the query plan. Look for:
- `BroadcastHashJoin`: broadcast join happening.
- `SortMergeJoin`: standard shuffle-based join.
- `BroadcastExchange`: the broadcast operation itself.

**Key config:**
- `spark.sql.autoBroadcastJoinThreshold`: max size for auto-broadcast. Default 10485760 (10 MB). Set to -1 to disable auto-broadcast.

In [0]:
# cross join
crossed = customers.crossJoin(orders)
print(f"Row count: {cross.count}")
crossed.show()
# display(crossed.orderBy("customer_id", "order_id"))

## Cross join

Produces the Cartesian product — every row on the left paired with every row on the right. Row count = left rows × right rows.

**Syntax:**
- `df1.crossJoin(df2)` — explicit cross join.
- `df1.join(df2)` with no condition — errors by default, requires `spark.sql.crossJoin.enabled = true` to allow implicitly.

**Safety guardrail:** Spark refuses implicit cross joins because they're usually a mistake. You must use `crossJoin()` explicitly to signal intent.

**When useful:**
- Combinatorial generation (every A × every B).
- Date-scaffolding: cross-joining entities with a date range to produce entity-date grid, then left-joining events.
- Rarely appropriate for typical relational queries.

**When to avoid:**
- Almost always in normal analytics work. If you're reaching for a cross join, verify it's actually what you want — usually a missing join condition is the real bug.

In [0]:
# union and union all
new_orders_data = [
    Row(order_id="1007", customer_id="C002", amount=310.00, order_date="2026-07-20"),
    Row(order_id="1008", customer_id="C005", amount=125.50, order_date="2026-07-21"),
]
new_orders = spark.createDataFrame(new_orders_data)

# Combine existing and new orders
combined = orders.union(new_orders)
print(f"Original: {orders.count()}, new: {new_orders.count()}, combined: {combined.count()}")
display(combined.orderBy("order_id"))

In [0]:
# unionByName matches columns by name, not by position
reordered = new_orders.select("customer_id", "order_id", "order_date", "amount")
try:
    orders.union(reordered)  # would produce wrong results
except Exception as e:
    print(f"union() error: {e}")

correct = orders.unionByName(reordered)
display(correct)

## Union operations

Combine rows from two DataFrames vertically. Not the same as joins, which combine columns horizontally.

**Operations:**
- `union(df2)` — appends rows. Requires identical schemas (columns, types, order). Does NOT deduplicate.
- `unionAll(df2)` — alias for `union()` in Spark 2.0+. Same behavior.
- `unionByName(df2)` — matches columns by name rather than position. Use when column orders might differ.
- `unionByName(df2, allowMissingColumns=True)` — allows one DataFrame to have columns the other doesn't; fills missing with nulls.

**Deduplication:** Neither `union` nor `unionAll` deduplicates in modern Spark. To remove duplicates after union, chain `.distinct()`:
```python
combined = df1.union(df2).distinct()
```

**Historical note:** In pre-2.0 Spark and in SQL, `UNION` deduplicated and `UNION ALL` didn't. Modern Spark's PySpark API doesn't follow the SQL convention — both are non-deduplicating.